# 5 折数据划分（训练/验证）

基于 `train_set_processed.csv` 进行 **5 折分层划分**：
- 每一折中，约 **1/5** 数据作为验证集
- 剩余 **4/5** 作为训练子集
- 按标签 `SeriousDlqin2yrs` 分层，尽量保持类别比例一致
- 将每一折的训练集与验证集保存到 `../data/processed/five_folds/`

In [22]:
import os
import pandas as pd
from sklearn.model_selection import StratifiedKFold

# 1) 读取处理后的训练数据（自动适配当前工作目录）
candidate_data_dirs = [
    "data/processed",
    "../data/processed",
]

data_processed_dir = None
for d in candidate_data_dirs:
    if os.path.exists(os.path.join(d, "train_set_processed.csv")):
        data_processed_dir = os.path.abspath(d)
        break

if data_processed_dir is None:
    raise FileNotFoundError("未找到 train_set_processed.csv，请检查 data/processed 路径")

input_path = os.path.join(data_processed_dir, "train_set_processed.csv")
out_dir = os.path.join(data_processed_dir, "five_folds")
os.makedirs(out_dir, exist_ok=True)

df = pd.read_csv(input_path, index_col=0)
target_col = "SeriousDlqin2yrs"

if target_col not in df.columns:
    raise ValueError(f"未找到目标列: {target_col}")

X = df.drop(columns=[target_col])
y = df[target_col]

print(f"当前工作目录: {os.getcwd()}")
print(f"数据目录: {data_processed_dir}")
print(f"数据读取成功: {input_path}")
print(f"总样本数: {len(df)}")
print(f"整体违约比例: {y.mean():.2%}")

当前工作目录: f:\graduation-project\Credit-card-delinquency-prediction-based-on-DNN\notebooks
数据目录: f:\graduation-project\Credit-card-delinquency-prediction-based-on-DNN\data\processed
数据读取成功: f:\graduation-project\Credit-card-delinquency-prediction-based-on-DNN\data\processed\train_set_processed.csv
总样本数: 119780
整体违约比例: 6.60%


In [23]:
# 2) 5 折分层划分：每折约 1/5 做验证集
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

summary = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    fold_train = df.iloc[train_idx].copy()
    fold_val = df.iloc[val_idx].copy()

    train_path = f"{out_dir}/fold_{fold}_train.csv"
    val_path = f"{out_dir}/fold_{fold}_val.csv"

    fold_train.to_csv(train_path, index=True)
    fold_val.to_csv(val_path, index=True)

    train_pos_rate = fold_train[target_col].mean()
    val_pos_rate = fold_val[target_col].mean()

    summary.append(
        {
            "fold": fold,
            "train_size": len(fold_train),
            "val_size": len(fold_val),
            "train_pos_rate": train_pos_rate,
            "val_pos_rate": val_pos_rate,
        }
    )

summary_df = pd.DataFrame(summary)
summary_df

,fold,train_size,val_size,train_pos_rate,val_pos_rate
0,1,95824,23956,0.065986,0.065954
1,2,95824,23956,0.065986,0.065954
2,3,95824,23956,0.065975,0.065996
3,4,95824,23956,0.065975,0.065996
4,5,95824,23956,0.065975,0.065996


In [24]:
# 3) 结果总览
print("5 折划分完成，文件已保存到:", out_dir)
print("\n各折统计（应接近 4:1）：")
print(summary_df.to_string(index=False, formatters={
    "train_pos_rate": "{:.4f}".format,
    "val_pos_rate": "{:.4f}".format,
}))

5 折划分完成，文件已保存到: f:\graduation-project\Credit-card-delinquency-prediction-based-on-DNN\data\processed\five_folds

各折统计（应接近 4:1）：
 fold  train_size  val_size train_pos_rate val_pos_rate
    1       95824     23956         0.0660       0.0660
    2       95824     23956         0.0660       0.0660
    3       95824     23956         0.0660       0.0660
    4       95824     23956         0.0660       0.0660
    5       95824     23956         0.0660       0.0660


In [25]:
# 4) 每折标准化：仅在训练子集 fit，保存参数，并用同参数变换训练/验证
from sklearn.preprocessing import StandardScaler

scaled_out_dir = os.path.join(data_processed_dir, "five_folds_standardized")
os.makedirs(scaled_out_dir, exist_ok=True)

scale_summary = []
for fold in range(1, 6):
    train_path = os.path.join(out_dir, f"fold_{fold}_train.csv")
    val_path = os.path.join(out_dir, f"fold_{fold}_val.csv")

    fold_train = pd.read_csv(train_path, index_col=0)
    fold_val = pd.read_csv(val_path, index_col=0)

    feature_cols = [c for c in fold_train.columns if c != target_col]

    # 1) 仅在训练集上拟合，得到均值和标准差
    scaler = StandardScaler()
    scaler.fit(fold_train[feature_cols])

    # 2) 保存该折标准化参数（每个特征的 mean/std）
    scaler_params = pd.DataFrame({
        "feature": feature_cols,
        "mean": scaler.mean_,
        "std": scaler.scale_,
    })
    scaler_params_path = os.path.join(scaled_out_dir, f"fold_{fold}_scaler_params.csv")
    scaler_params.to_csv(scaler_params_path, index=False)

    # 3) 用同一组参数标准化训练集和验证集
    X_train_scaled = scaler.transform(fold_train[feature_cols])
    X_val_scaled = scaler.transform(fold_val[feature_cols])

    fold_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=fold_train.index)
    fold_train_scaled[target_col] = fold_train[target_col].values

    fold_val_scaled = pd.DataFrame(X_val_scaled, columns=feature_cols, index=fold_val.index)
    fold_val_scaled[target_col] = fold_val[target_col].values

    train_scaled_path = os.path.join(scaled_out_dir, f"fold_{fold}_train_scaled.csv")
    val_scaled_path = os.path.join(scaled_out_dir, f"fold_{fold}_val_scaled.csv")

    fold_train_scaled.to_csv(train_scaled_path, index=True)
    fold_val_scaled.to_csv(val_scaled_path, index=True)

    scale_summary.append({
        "fold": fold,
        "train_size": len(fold_train_scaled),
        "val_size": len(fold_val_scaled),
        "train_pos_rate": fold_train_scaled[target_col].mean(),
        "val_pos_rate": fold_val_scaled[target_col].mean(),
    })

scale_summary_df = pd.DataFrame(scale_summary)
print("标准化处理完成，文件已保存到:", scaled_out_dir)
print("（每折仅在训练集 fit，并将同参数应用于训练集和验证集）")
print()
print(scale_summary_df.to_string(index=False, formatters={
    "train_pos_rate": "{:.4f}".format,
    "val_pos_rate": "{:.4f}".format,
}))

标准化处理完成，文件已保存到: f:\graduation-project\Credit-card-delinquency-prediction-based-on-DNN\data\processed\five_folds_standardized
（每折仅在训练集 fit，并将同参数应用于训练集和验证集）

 fold  train_size  val_size train_pos_rate val_pos_rate
    1       95824     23956         0.0660       0.0660
    2       95824     23956         0.0660       0.0660
    3       95824     23956         0.0660       0.0660
    4       95824     23956         0.0660       0.0660
    5       95824     23956         0.0660       0.0660


## 过采样
- 只对每折训练集做过采样。
- 验证集不做过采样，保持原始类别分布。
- 优先使用 SMOTE（本次实际运行就是 SMOTE）。

In [26]:
# 5) 每折过采样：参数可配置，仅训练集过采样，验证时直接使用标准化验证集
from collections import Counter
from sklearn.utils import resample

# ===================== 可配置参数区域 =====================
# 目标比例设置：多数类:少数类 = TARGET_MAJORITY_MINORITY_RATIO[0] : TARGET_MAJORITY_MINORITY_RATIO[1]
# 例如 9:1 表示少数类样本数应达到多数类的 1/9
TARGET_MAJORITY_MINORITY_RATIO = (9, 1)
USE_SMOTE = True
SAVE_OUTPUTS = True
RANDOM_STATE = 42
SCALED_DATA_DIR = os.path.join(data_processed_dir, "five_folds_standardized")
OUTPUT_DIR = os.path.join(data_processed_dir, "five_folds_oversampled")
# =========================================================

# 动态计算少数类目标比例（少数类样本数 / 多数类样本数）
target_minority_ratio = (
    TARGET_MAJORITY_MINORITY_RATIO[1] / TARGET_MAJORITY_MINORITY_RATIO[0]
)

# 检查 imbalanced-learn 可用性
try:
    from imblearn.over_sampling import SMOTE
    smote_available = True
except ImportError:
    smote_available = False
    print("未检测到 imbalanced-learn，将自动回退到随机过采样。")


def oversample_single_fold(
    fold_number,
    input_dir,
    output_dir,
    target_ratio,
    target_col_name,
    use_smote=True,
    save=True,
    random_state=42,
):
    """对单个 fold 的训练集进行过采样，并返回过采样前后统计信息。"""
    train_path = os.path.join(input_dir, f"fold_{fold_number}_train_scaled.csv")
    train_df = pd.read_csv(train_path, index_col=0)

    feature_cols = [column for column in train_df.columns if column != target_col_name]
    X_train = train_df[feature_cols]
    y_train = train_df[target_col_name]
    before_counts = Counter(y_train)

    method_used = "Unknown"
    if use_smote and smote_available:
        sampler = SMOTE(sampling_strategy=target_ratio, random_state=random_state)
        X_resampled, y_resampled = sampler.fit_resample(X_train, y_train)
        method_used = "SMOTE"
    else:
        train_joined = train_df.copy()
        class_counts = train_joined[target_col_name].value_counts()
        majority_label = class_counts.idxmax()
        minority_label = class_counts.idxmin()

        majority_df = train_joined[train_joined[target_col_name] == majority_label]
        minority_df = train_joined[train_joined[target_col_name] == minority_label]

        target_minority_count = int(len(majority_df) * target_ratio)
        if target_minority_count < len(minority_df):
            target_minority_count = len(minority_df)

        minority_upsampled = resample(
            minority_df,
            replace=True,
            n_samples=target_minority_count,
            random_state=random_state,
        )

        resampled_df = pd.concat([majority_df, minority_upsampled], axis=0)
        resampled_df = resampled_df.sample(frac=1, random_state=random_state)

        X_resampled = resampled_df[feature_cols]
        y_resampled = resampled_df[target_col_name]
        method_used = "RandomOverSampling"

    train_resampled_df = pd.DataFrame(X_resampled, columns=feature_cols)
    train_resampled_df[target_col_name] = y_resampled
    after_counts = Counter(train_resampled_df[target_col_name])

    if save:
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, f"fold_{fold_number}_train_oversampled.csv")
        train_resampled_df.to_csv(output_path, index=False)

    return {
        "fold": fold_number,
        "method": method_used,
        "train_before_0": before_counts.get(0, 0),
        "train_before_1": before_counts.get(1, 0),
        "train_after_0": after_counts.get(0, 0),
        "train_after_1": after_counts.get(1, 0),
        "after_ratio_1_to_0": round(after_counts.get(1, 0) / after_counts.get(0, 1), 4),
    }


os.makedirs(OUTPUT_DIR, exist_ok=True)

summary_results = []
for fold_idx in range(1, 6):
    fold_stats = oversample_single_fold(
        fold_number=fold_idx,
        input_dir=SCALED_DATA_DIR,
        output_dir=OUTPUT_DIR,
        target_ratio=target_minority_ratio,
        target_col_name=target_col,
        use_smote=USE_SMOTE,
        save=SAVE_OUTPUTS,
        random_state=RANDOM_STATE,
    )
    summary_results.append(fold_stats)

oversample_summary_df = pd.DataFrame(summary_results)
print("过采样完成！配置信息：")
print(
    f"目标比例（多数类:少数类）: "
    f"{TARGET_MAJORITY_MINORITY_RATIO[0]}:{TARGET_MAJORITY_MINORITY_RATIO[1]}"
)
print(f"是否使用SMOTE: {USE_SMOTE and smote_available}")
print(f"是否写入文件: {SAVE_OUTPUTS}")
print(f"标准化数据目录: {SCALED_DATA_DIR}")
print(f"过采样输出目录: {OUTPUT_DIR}")
print("\n详细统计：")
print(oversample_summary_df.to_string(index=False))

过采样完成！配置信息：
目标比例（多数类:少数类）: 9:1
是否使用SMOTE: True
是否写入文件: True
标准化数据目录: f:\graduation-project\Credit-card-delinquency-prediction-based-on-DNN\data\processed\five_folds_standardized
过采样输出目录: f:\graduation-project\Credit-card-delinquency-prediction-based-on-DNN\data\processed\five_folds_oversampled

详细统计：
 fold method  train_before_0  train_before_1  train_after_0  train_after_1  after_ratio_1_to_0
    1  SMOTE           89501            6323          89501           9944              0.1111
    2  SMOTE           89501            6323          89501           9944              0.1111
    3  SMOTE           89502            6322          89502           9944              0.1111
    4  SMOTE           89502            6322          89502           9944              0.1111
    5  SMOTE           89502            6322          89502           9944              0.1111
